In [2]:
!pip install statsmodels

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.6 MB 1.3 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/9.6 MB 1.3 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/9.6 MB 1.3 MB/s eta 0:00:07
   ----- ---------------------------------- 1.3/9.6 MB 1.3 MB/s eta 0:00:07
   ------ --------------------------------- 1.6/9.6 MB 1.3 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.6 MB 1.3 MB/s eta 0:00:07
   -------- ------------------------------- 2.1/9.6 MB 1.3 MB/s eta 0:00:06
   --------- ------------------------------ 2.4/9.6 MB 1.3 MB/s eta 0:00:06
   ---------- ----------------------------- 2.6/9.6 MB 1.3 MB/s eta 0:00:06
   ------------ --------------------------- 2.9/9.6 MB 1.3 MB/s eta 0:00:06
   ------------- -------------------------- 3.1/9.6 MB 1.3 MB/s eta 0:00:06
   -------------- --------


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install pmdarima

   ---------------------------------------- 0.0/625.1 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/625.1 kB ? eta -:--:--
   ---------------------------------------- 625.1/625.1 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 1.4 MB/s eta 0:00:02
   ----------- ---------------------------- 0.8/2.7 MB 1.3 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.7 MB 1.3 MB/s eta 0:00:02
   ------------------- -------------------- 1.3/2.7 MB 1.3 MB/s eta 0:00:02
   ----------------------- ---------------- 1.6/2.7 MB 1.3 MB/s eta 0:00:01
   --------------------------- ------------ 1.8/2.7 MB 1.3 MB/s eta 0:00:01
   ------------------------------- -------- 2.1/2.7 MB 1.3 MB/s eta 0:00:01
   ---------------------------------- ----- 2.4/2.7 MB 1.3 MB/s eta 0:00:01
   ---------------------------


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ======================
# 1. Imports
# ======================
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
from pmdarima import auto_arima

import warnings
warnings.filterwarnings("ignore")  # To suppress ARIMA fitting warnings

# Load dataset
df = pd.read_csv("EcoTrack_Waste_Bins_Dataset.csv")

# Parse dates
df["Start_Date"] = pd.to_datetime(df["Start_Date"], format="%d-%m-%Y", errors="coerce")
df["End_Date(100%)"] = pd.to_datetime(df["End_Date(100%)"], format="%d-%m-%Y", errors="coerce")
df["Fullness_80%_Date"] = pd.to_datetime(df["Fullness_80%_Date"], format="%d-%m-%Y", errors="coerce")


In [3]:
print(df.head())

   Bin_ID  Longitude   Latitude Location_Name    Zone_Type Start_Date  \
0  B00001  77.629875  12.943689       MG Road  Residential 2025-01-01   
1  B00001  77.629875  12.943689       MG Road  Residential 2025-01-06   
2  B00001  77.629875  12.943689       MG Road  Residential 2025-01-12   
3  B00001  77.629875  12.943689       MG Road  Residential 2025-01-17   
4  B00001  77.629875  12.943689       MG Road  Residential 2025-01-24   

  Fullness_80%_Date End_Date(100%)  Fill_Duration(days)  
0        2025-01-03     2025-01-05                    4  
1        2025-01-09     2025-01-11                    5  
2        2025-01-14     2025-01-16                    4  
3        2025-01-21     2025-01-23                    6  
4        2025-01-28     2025-01-30                    6  


In [4]:
# -----------------------
# Train per-bin model
# -----------------------
def train_arima_for_bin(bin_id):
    """Train Auto ARIMA on Fill Duration of one bin."""
    g = df[df["Bin_ID"] == bin_id].sort_values("Start_Date")
    series = g["Fill_Duration(days)"].astype(float).rolling(window=3, min_periods=1).mean()


    if len(series) < 5:  # not enough data
        return None

    try:
        # Auto ARIMA automatically selects (p,d,q)
        model_fit = auto_arima(series, seasonal=False, stepwise=True, suppress_warnings=True)
        return model_fit
    except Exception as e:
        print(f"⚠️ ARIMA failed for {bin_id}: {e}")
        return None



In [5]:
def predict_for_location_zone(start_date_str, location_name, zone_type):
    start_dt = datetime.strptime(start_date_str, "%d-%m-%Y")
    
    # Filter bins in location and zone
    subset = df[(df["Location_Name"] == location_name) & (df["Zone_Type"] == zone_type)]
    unique_bins = subset["Bin_ID"].unique().tolist()

    # Zone-level time series (for fallback)
    zone_series = df[df["Zone_Type"] == zone_type]["Fill_Duration(days)"].groupby(df["Start_Date"]).mean()

    # Zone average duration (fallback)
    zone_avg = int(round(zone_series.mean()))

    results = []
    for bin_id in unique_bins:
        model_fit = train_arima_for_bin(bin_id)

        if model_fit is None:
            pred_days = zone_avg
        else:
            forecast = model_fit.predict(n_periods=1)
            forecast_value = float(np.array(forecast)[0])
            pred_days = max(1, int(round(forecast_value)))

        # Predicted dates
        pred_80 = start_dt + timedelta(days=max(1, pred_days - 2))
        pred_100 = start_dt + timedelta(days=pred_days)

        results.append({
            "Bin_ID": bin_id,
            "Start_Date": start_date_str,
            "Location_Name": location_name,
            "Zone_Type": zone_type,
            "Predicted_Fill_Duration(days)": pred_days,
            "Predicted_80pct_Date": pred_80.strftime("%d-%m-%Y"),
            "Predicted_100pct_Date": pred_100.strftime("%d-%m-%Y")
        })

    return pd.DataFrame(results)


In [6]:
# ======================
# 4. Visualization Functions (Updated: Compare Actual vs Predicted)
# ======================

# Plot predictions for all bins in a location/zone (Actual vs Predicted comparison - Line Chart)
def plot_predictions(start_date_str, location, zone):
    out = predict_for_location_zone(start_date_str, location, zone)
    if out.empty:
        print("No predictions available.")
        return

    # Compute actuals (last observed durations)
    actuals = []
    for b in out["Bin_ID"]:
        g = df[df["Bin_ID"] == b].sort_values("Start_Date")
        if not g.empty:
            actuals.append(g["Fill_Duration(days)"].values[-1])  # last actual duration
        else:
            actuals.append(None)
    out["Actual_Fill_Duration(days)"] = actuals

    # Line chart: Actual vs Predicted
    plt.figure(figsize=(10,5))
    x = range(len(out["Bin_ID"]))
    plt.plot(x, out["Actual_Fill_Duration(days)"], marker="o", label="Actual", color="blue")
    plt.plot(x, out["Predicted_Fill_Duration(days)"], marker="x", linestyle="--", label="Predicted", color="red")

    plt.xticks(x, out["Bin_ID"], rotation=90)
    plt.title(f"Actual vs Predicted Fill Durations - {location} ({zone})")
    plt.xlabel("Bin_ID")
    plt.ylabel("Fill Duration (days)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return out  # optional: return dataframe



In [7]:
import ipywidgets as widgets
from ipywidgets import interact, interact_manual
from IPython.display import display, clear_output

# Dropdowns for location and zone type
location_dropdown = widgets.Dropdown(
    options=sorted(df["Location_Name"].unique()),
    description="Location:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="50%")
)

zone_dropdown = widgets.Dropdown(
    options=sorted(df["Zone_Type"].unique()),
    description="Zone Type:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="50%")
)

# Date input (text box in DD-MM-YYYY format)
date_input = widgets.Text(
    value="01-01-2025",
    placeholder="Enter date (DD-MM-YYYY)",
    description="Start Date:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="50%")
)

# Button to trigger prediction
button = widgets.Button(description="Predict Fill Dates", button_style="success")

# Output area
output = widgets.Output()

# Function to run when button clicked
def on_button_click(b):
    with output:
        clear_output()
        start_date = date_input.value
        location = location_dropdown.value
        zone = zone_dropdown.value
        
        # Run prediction
        preds = predict_for_location_zone(start_date, location, zone)
        if preds.empty:
            print("⚠️ No predictions available for this selection.")
            return
        
        # Show table
        display(preds)
        
        # Show visualization
        plot_predictions(start_date, location, zone)
        

# Connect button to function
button.on_click(on_button_click)

# Display UI
display(location_dropdown, zone_dropdown, date_input, button, output)


Dropdown(description='Location:', layout=Layout(width='50%'), options=('BTM Layout', 'Banashankari', 'Basavana…

Dropdown(description='Zone Type:', layout=Layout(width='50%'), options=('Commercial', 'Industrial', 'Market', …

Text(value='01-01-2025', description='Start Date:', layout=Layout(width='50%'), placeholder='Enter date (DD-MM…

Button(button_style='success', description='Predict Fill Dates', style=ButtonStyle())

Output()

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_overall_model(df):
    actual = df["Fill_Duration(days)"].values
    
    # Use zone-wise averages as predictions
    predicted = []
    for _, row in df.iterrows():
        zone_subset = df[(df["Location_Name"] == row["Location_Name"]) &
                         (df["Zone_Type"] == row["Zone_Type"])]
        zone_avg = int(round(zone_subset["Fill_Duration(days)"].mean()))
        predicted.append(zone_avg)
    
    predicted = np.array(predicted)
    
    # Metrics
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    accuracy = 100 - (mae / np.mean(actual) * 100)  # percentage accuracy
    
    print(" Overall Model Evaluation")
    print(f"Mean Absolute Error (MAE): {mae:.2f} days")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f} days")
    print(f"Approximate Accuracy: {accuracy:.2f}%")

evaluate_overall_model(df)


 Overall Model Evaluation
Mean Absolute Error (MAE): 1.06 days
Root Mean Squared Error (RMSE): 1.31 days
Approximate Accuracy: 79.28%
